# AgentCacheBench: M2 Minimal Falsification Pilot

[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshirsagarps/agent-cache-bench/blob/main/notebooks/M2_falsification_pilot.ipynb)

This notebook executes **Milestone 2 (M2 Minimal Falsification Pilot)** of AgentCacheBench on Google Colab GPU runtimes.

### Primary Research Question
> *Do conventional request-level cache-hit, prefix-overlap, or logical reuse metrics accurately predict actual computation avoided and session-level efficiency in realistic stateful LLM-agent trajectories?*

### Policy Enforcement
In accordance with the **COLAB EXECUTION POLICY**, this notebook captures the exact GPU assigned (`gpu_name`, `gpu_memory_gb`, `cuda_version`, `pytorch_version`), records raw trial JSON observations, and evaluates scenarios S0–S4.

In [ ]:
# Step 1: Clone Repository and Install AgentCacheBench
import os, sys, subprocess

if not os.path.exists('agentcachebench'):
    !git clone https://github.com/kshirsagarps/agent-cache-bench.git
    %cd agent-cache-bench

!pip install -q numpy scipy pandas jsonschema pyyaml matplotlib pillow

import torch
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Assigned: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("Running on CPU/MPS runtime mode.")

In [ ]:
# Step 2: Import AgentCacheBench Engine & Colab Sync Utilities
from agentcachebench.runner.engine import BenchmarkRunner
from agentcachebench.runner.colab_sync import is_google_colab, mount_google_drive, get_colab_gpu_provenance, save_colab_checkpoint
from agentcachebench.workloads.tool_use import generate_tool_use_trajectory
from agentcachebench.workloads.coding import generate_coding_trajectory
from agentcachebench.scenarios.mutations import apply_block_shift
from agentcachebench.metrics.calculators import compute_correlation_rho_and_r

gpu_info = get_colab_gpu_provenance()
print(f"Loaded Colab GPU Provenance: {gpu_info}")

In [ ]:
# Step 3: Run M2 Falsification Pilot Scenarios (S0 - S4)
runner = BenchmarkRunner(output_dir="results/raw")

t_s0 = generate_tool_use_trajectory(num_steps=5, pause_ms=0.0, seed=40)
t_s1 = generate_tool_use_trajectory(num_steps=5, pause_ms=100.0, seed=41)
t_s2 = generate_tool_use_trajectory(num_steps=5, pause_ms=1000.0, seed=42)

t_s3_raw = generate_coding_trajectory(num_steps=5, mutation_type="middle_edit", seed=43)
t_s3 = []
for step in t_s3_raw:
    tokens = step["prompt_tokens"]
    if step["step_id"] in [2, 4]:
        tokens = apply_block_shift(tokens, shift_size=3)
    t_s3.append({"step_id": step["step_id"], "prompt_tokens": tokens, "pause_ms": 1000.0, "event_type": "front_shift"})

t_s4 = generate_coding_trajectory(num_steps=5, mutation_type="file_replace", seed=44)
for step in t_s4:
    step["pause_ms"] = 30000.0

scenarios = [
    ("m2_colab_s0", "S0", "B0", t_s0, {"enable_pause_decay": False}),
    ("m2_colab_s1", "S1", "B1", t_s1, {"enable_pause_decay": False}),
    ("m2_colab_s2", "S2", "B1", t_s2, {"enable_pause_decay": False}),
    ("m2_colab_s3", "S3", "B1", t_s3, {"enable_pause_decay": False}),
    ("m2_colab_s4", "S4", "B3", t_s4, {"enable_pause_decay": True, "max_cache_blocks": 128}),
]

pilot_results = []
all_overlaps, all_avoideds = [], []
mutated_overlaps, mutated_avoideds = [], []

for exp_id, scenario_id, baseline_id, trajectory, cfg in scenarios:
    res = runner.run_experiment(exp_id, "pilot_workload", trajectory, scenario_id, baseline_id, cfg)
    pilot_results.append(res)
    for st in res["steps"]:
        all_overlaps.append(st["logical_overlap"])
        all_avoideds.append(st["actual_compute_avoided"])
        if scenario_id in ["S3", "S4"]:
            mutated_overlaps.append(st["logical_overlap"])
            mutated_avoideds.append(st["actual_compute_avoided"])

overall_rho, _ = compute_correlation_rho_and_r(all_overlaps, all_avoideds)
mutated_rho, _ = compute_correlation_rho_and_r(mutated_overlaps, mutated_avoideds)

print(f"=== M2 PILOT COMPLETED ON COLAB ===")
print(f"Overall Spearman Rho: {overall_rho:.4f}")
print(f"Mutated Context Spearman Rho: {mutated_rho:.4f}")

In [ ]:
# Step 4: Save Decision & Summary JSONs
decision = "GO" if mutated_rho < 0.70 else "NO_GO"
go_no_go_data = {
    "milestone": "M2",
    "decision": decision,
    "rationale": f"Under context mutation and pause decay (S3/S4), Spearman rho dropped to {mutated_rho:.4f} (< 0.70 threshold).",
    "mutated_spearman_rho": mutated_rho,
    "threshold": 0.70,
    "gpu_provenance": gpu_info
}

with open("results/m2_go_no_go.json", "w") as f:
    json.dump(go_no_go_data, f, indent=2)

print(f"M2 Decision: {decision}")